# My tables: studios, people, credits, reviewers, consumer reviews

**Author: Paulius Sivickas**

Database Management, AUAS — individual contribution.

Builds into PostgreSQL database `Database_Assignemtnt_DDB` using the **group schema** (`schema.sql`, v3.3, written by Anguelito).

Run the cells top to bottom. Cell 2 asks for your Postgres password once; it is not saved in the file.

**Files needed next to this notebook:** `schema.sql` and the `data/` folder with the spreadsheets.

## 1. Packages

In [7]:
%pip install psycopg2-binary openpyxl --quiet
print("done")

Note: you may need to restart the kernel to use updated packages.
done


## 2. Connect

`getpass` hides the password and keeps it out of the saved notebook, so nothing secret ends up on GitHub.

In [8]:
# Author: Paulius Sivickas
import os, re, getpass
import pandas as pd
import psycopg2

PASSWORD = getpass.getpass("Postgres password: ")

con = psycopg2.connect(
    dbname="Database_Assignemtnt_DDB",
    user="postgres",
    password=PASSWORD,
    host="localhost",
    port="5432",
)
cur = con.cursor()
cur.execute("SELECT version();")
print(cur.fetchone()[0])

PostgreSQL 18.6 on aarch64-apple-darwin24.6.0, compiled by Apple clang version 17.0.0 (clang-1700.0.13.5), 64-bit


## 3. Check the data files

In [9]:
# Author: Paulius Sivickas
DATA = "data"
files = {
    "meta":   os.path.join(DATA, "metaClean43Brightspace.xlsx"),
    "sales":  os.path.join(DATA, "sales.xlsx"),
    "user":   os.path.join(DATA, "UserReviewsClean43LIWC.xlsx"),
    "expert": os.path.join(DATA, "ExpertReviewsClean43LIWC.xlsx"),
}
for k, p in files.items():
    print(("  OK      " if os.path.exists(p) else "  MISSING "), k, "->", p)

HAVE_USER_REVIEWS = os.path.exists(files["user"])
if not HAVE_USER_REVIEWS:
    print("\nUserReviews file not present - the consumer_reviews TABLE will still be created, just left empty.")
print(("  OK      " if os.path.exists("schema.sql") else "  MISSING "), "schema.sql in the notebook folder")

  OK       meta -> data/metaClean43Brightspace.xlsx
  OK       sales -> data/sales.xlsx
  OK       user -> data/UserReviewsClean43LIWC.xlsx
  OK       expert -> data/ExpertReviewsClean43LIWC.xlsx
  OK       schema.sql in the notebook folder


## 4. Create the tables (group schema)

The tables are **not written in this notebook any more**. This cell runs the group's `schema.sql`, so everyone builds the exact same database.

`schema.sql` drops tables in the group's order, but my older tables (`directors`, `cast_members`, `movie_cast`, `user_reviews`, ...) are not in its list, and they point at `movies`. So the first step removes every old and new table with `CASCADE`, then the group script runs on a clean slate.

**My tables in the group schema:** `studios`, `movie_studios`, `people`, `movie_credits`, `reviewers`, `consumer_reviews`. `movies` is loaded too because they all point at it with foreign keys.

Two design points I can defend:

- `studios` is its own table because the studio name repeated on every movie row is a **3NF** problem (the name depends on the studio, not on the movie). A film can have two studios, so the link is the bridge table `movie_studios`.
- `cast` and `director` hold up to 20 comma-separated names in one cell, which breaks **1NF**. They become one `people` table plus the bridge `movie_credits`, tagged with `credit_type` (`director` or `cast`). One table means an actor-director is stored once.

In [10]:
# Author: Paulius Sivickas
SCHEMA_FILE = "schema.sql"           # the group's file, saved next to this notebook

# 1) clear every old and new table (CASCADE = also remove anything pointing at it)
ALL_TABLES = ["movie_credits", "movie_studios", "movie_genres", "consumer_reviews",
              "expert_reviews", "reviewers", "people", "studios", "genres", "movies",
              # my older tables from the previous schema
              "movie_cast", "movie_studio", "movie_director", "user_reviews",
              "cast_members", "directors"]
for t in ALL_TABLES:
    cur.execute(f"DROP TABLE IF EXISTS {t} CASCADE")
con.commit()

# 2) run the group's schema file exactly as written
with open(SCHEMA_FILE, encoding="utf-8") as f:
    cur.execute(f.read())
con.commit()

cur.execute("""SELECT table_name FROM information_schema.tables
               WHERE table_schema='public' ORDER BY table_name;""")
for (t,) in cur.fetchall():
    print("created:", t)

def has_col(table, col):
    cur.execute("""SELECT 1 FROM information_schema.columns
                   WHERE table_name = %s AND column_name = %s""", (table, col))
    return cur.fetchone() is not None

# billing_order and review_date are loaded only if the group schema has those columns
HAVE_BILLING     = has_col("movie_credits", "billing_order")
HAVE_REVIEW_DATE = has_col("consumer_reviews", "review_date")
print("\nmovie_credits.billing_order   :", "yes" if HAVE_BILLING else "not in schema.sql - skipped")
print("consumer_reviews.review_date  :", "yes" if HAVE_REVIEW_DATE else "not in schema.sql - skipped")

created: consumer_reviews
created: expert_reviews
created: genres
created: movie_credits
created: movie_genres
created: movie_studios
created: movies
created: people
created: reviewers
created: studios

movie_credits.billing_order   : not in schema.sql - skipped
consumer_reviews.review_date  : not in schema.sql - skipped


## 5. Little helpers

In [11]:
# Author: Paulius Sivickas
# strip only ordinary whitespace. Python's plain .strip() also removes the invisible
# no-break space (\xa0), which is the second half of letters like "à" in garbled text,
# so "CerdÃ\xa0" would lose its ending and could never be repaired.
WS = " \t\r\n"
from datetime import datetime
SUFFIXES = {"jr", "jr.", "sr", "sr.", "ii", "iii", "iv"}

def fix_mojibake(raw):
    """Mojibake = garbled text such as 'KormÃ¡kur' for 'Kormákur': correct UTF-8 bytes
    that were read as Latin-1. Encoding back to Latin-1 and decoding as UTF-8 undoes it.
    If that fails the text was fine (or damaged beyond repair), so it is left alone."""
    if raw is None:
        return raw
    s = str(raw)
    try:
        return s.encode("latin1").decode("utf-8")
    except (UnicodeDecodeError, UnicodeEncodeError):
        return s

def clean_text(v):
    """Strip the stray quotes and spaces the scraper left behind."""
    if v is None: return None
    s = str(v).strip(WS)
    if s in ("None", "nan", ""): return None
    return s.strip("'\"").strip(WS) or None

def split_list(v):
    """'Drama,Crime' -> ['Drama','Crime']   (plain comma split, used for studios)"""
    s = clean_text(v)
    return [p.strip(WS) for p in s.split(",") if p.strip(WS)] if s else []

def split_names(v):
    """Suffix-aware split for people. 'Michael Landon, Jr.' stays ONE name instead of
    becoming 'Michael Landon' and 'Jr.'. Also repairs mojibake and removes repeats."""
    names = []
    for part in split_list(v):
        if part.lower() in SUFFIXES and names:
            names[-1] = names[-1] + ", " + part      # glue the suffix back on
        else:
            names.append(part)
    out, seen = [], set()
    for n in names:
        n = fix_mojibake(n)
        if n not in seen:
            seen.add(n); out.append(n)
    return out

def split_studios(v):
    """Studios: plain comma split + mojibake repair + no repeats."""
    out, seen = [], set()
    for n in split_list(v):
        n = fix_mojibake(n)
        if n not in seen:
            seen.add(n); out.append(n)
    return out

def norm_title(t):
    """Lowercase, drop a trailing '(S-Korea)', strip punctuation. Used to match sales to metaClean."""
    if t is None or pd.isna(t): return None
    s = re.sub(r"\s*\(.*?\)\s*$", "", str(t).lower().strip())
    return re.sub(r"[^a-z0-9]+", "", s) or None

def num(v):
    try:
        if v is None or (isinstance(v, float) and pd.isna(v)): return None
        return float(v)
    except (TypeError, ValueError):
        return None

def to_int(v):
    """Whole-number columns (INT in the schema) must not receive 17.0 or 17.6."""
    x = num(v)
    return None if x is None else int(round(x))

def parse_date(v):
    """'Apr 25, 2013' -> date(2013, 4, 25). Unreadable or missing -> None."""
    if v is None: return None
    if hasattr(v, "date"): return v.date()          # already a datetime
    s = clean_text(v)
    if not s: return None
    for fmt in ("%b %d, %Y", "%B %d, %Y", "%Y-%m-%d"):
        try: return datetime.strptime(s, fmt).date()
        except ValueError: pass
    d = pd.to_datetime(s, errors="coerce")
    return None if pd.isna(d) else d.date()

def no_nul(s):
    """PostgreSQL refuses the NUL character (0x00) inside text."""
    return s.replace("\x00", "") if isinstance(s, str) else s

print("helpers ready")
print(split_names("Michael Landon, Jr., Someone Else"))

helpers ready
['Michael Landon, Jr.', 'Someone Else']


## 6. Read the spreadsheets

In [12]:
# Author: Paulius Sivickas
meta  = pd.read_excel(files["meta"])
sales = pd.read_excel(files["sales"], sheet_name="numbers253")
sales = sales.drop(columns=[c for c in sales.columns if str(c).startswith("Unnamed")])

print("metaClean:", meta.shape)
print("sales    :", sales.shape)

sales["title_norm"] = sales["title"].map(norm_title)
sales_best = (sales.sort_values("worldwide_box_office", ascending=False)
                   .drop_duplicates("title_norm", keep="first")
                   .set_index("title_norm"))
print("unique sales titles:", len(sales_best))

metaClean: (11364, 13)
sales    : (30612, 15)
unique sales titles: 29905


## 7. studios, movies and movie_studios — **my tables**

One row per studio, plus the bridge `movie_studios` that links each film to **all** of its studios.

About 1.7% of films list more than one studio (co-productions), and the bridge keeps every one of them.

**movies** are loaded in the same cell because every one of my tables points at them with a foreign key.

`box_office` in the group schema is one column, so one measure has to be chosen. It is set in a single variable (`BOX_OFFICE_COLUMN`) so the whole group can change it in one place. Default: worldwide. Domestic has more films with a value (about 81% against 65%), worldwide matches the budget better — that choice belongs to the group, not to this cell.

In [13]:
# Author: Paulius Sivickas
from psycopg2.extras import execute_values

BOX_OFFICE_COLUMN = "worldwide_box_office"     # or "domestic_box_office" — group decision

# ---- studios -------------------------------------------------------------
studio_names = sorted({n for v in meta["studio"].dropna() for n in split_studios(v)})
execute_values(cur, "INSERT INTO studios (studio_name) VALUES %s ON CONFLICT DO NOTHING",
               [(n,) for n in studio_names])
con.commit()
cur.execute("SELECT studio_name, studio_id FROM studios")
studio_ids = dict(cur.fetchall())

# ---- movies --------------------------------------------------------------
movie_rows, seen_urls = [], set()
for r in meta.itertuples(index=False):
    url   = clean_text(r.url)
    title = clean_text(r.title)
    if not url or not title or url in seen_urls:
        continue                                   # title and url are NOT NULL in the schema
    seen_urls.add(url)

    tn = norm_title(r.title)
    s  = sales_best.loc[tn] if tn in sales_best.index else None
    movie_rows.append((
        title, url,
        num(s["production_budget"])  if s is not None else None,
        num(s[BOX_OFFICE_COLUMN])    if s is not None else None,
        to_int(r.runtime),
    ))

execute_values(cur, """INSERT INTO movies
    (title, metacritic_url, production_budget, box_office, runtime_minutes) VALUES %s""",
    movie_rows, page_size=2000)
con.commit()

# the database hands out movie_id (SERIAL), so read the ids back by url
cur.execute("SELECT metacritic_url, movie_id FROM movies")
url_to_id = dict(cur.fetchall())

# ---- movie_studios (bridge) ----------------------------------------------
ms_rows = set()
for r in meta.itertuples(index=False):
    mid = url_to_id.get(clean_text(r.url))
    if mid is None: continue
    for name in split_studios(r.studio):
        ms_rows.add((mid, studio_ids[name]))

execute_values(cur, "INSERT INTO movie_studios (movie_id, studio_id) VALUES %s ON CONFLICT DO NOTHING",
               sorted(ms_rows))
con.commit()

print("studios       :", len(studio_ids))
print("movies        :", len(movie_rows))
print("  with budget       :", sum(1 for m in movie_rows if m[2] is not None))
print("  with box office   :", sum(1 for m in movie_rows if m[3] is not None), f"({BOX_OFFICE_COLUMN})")
print("movie_studios :", len(ms_rows))

studios       : 1118
movies        : 11364
  with budget       : 3847
  with box office   : 5952 (worldwide_box_office)
movie_studios : 11206


## 8. people and movie_credits — **my tables**

`cast` has commas in 94.2% of rows (up to 20 names in one cell), `director` in 0.03%. That is the 1NF fix. `billing_order` is the position in the cast list (1 = top-billed) and is filled only if `schema.sql` has that column.

Three cleaning steps, all from the group's schema notes:

1. **Suffix-aware split** — `Michael Landon, Jr.` is one person, not "Michael Landon" and "Jr.".
2. **Mojibake repair** — `KormÃ¡kur` becomes `Kormákur`.
3. **One table for everybody** — `people` has each name once; `movie_credits` says whether the credit is `director` or `cast`.

In [14]:
# Author: Paulius Sivickas
credits = {}                        # (movie_id, person_name, credit_type) -> billing_order
for r in meta.itertuples(index=False):
    mid = url_to_id.get(clean_text(r.url))
    if mid is None: continue
    for name in split_names(r.director):
        credits[(mid, name, "director")] = None          # no billing order for directors
    for order, name in enumerate(split_names(r.cast), start=1):
        credits[(mid, name, "cast")] = order             # 1 = first name in the cast list

people_names = sorted({name for _, name, _ in credits})
execute_values(cur, "INSERT INTO people (person_name) VALUES %s ON CONFLICT DO NOTHING",
               [(n,) for n in people_names], page_size=5000)
con.commit()
cur.execute("SELECT person_name, person_id FROM people")
person_ids = dict(cur.fetchall())

credit_rows = [(mid, person_ids[name], ctype, order)
               for (mid, name, ctype), order in credits.items()]
if HAVE_BILLING:
    execute_values(cur, """INSERT INTO movie_credits (movie_id, person_id, credit_type, billing_order)
                           VALUES %s ON CONFLICT DO NOTHING""", credit_rows, page_size=5000)
else:
    execute_values(cur, """INSERT INTO movie_credits (movie_id, person_id, credit_type)
                           VALUES %s ON CONFLICT DO NOTHING""",
                   [c[:3] for c in credit_rows], page_size=5000)
con.commit()

print("people        :", len(person_ids))
print("movie_credits :", len(credit_rows))
print("  director credits:", sum(1 for c in credit_rows if c[2] == "director"))
print("  cast credits    :", sum(1 for c in credit_rows if c[2] == "cast"))

people        : 30314
movie_credits : 59633
  director credits: 11350
  cast credits    : 48283


## 9. reviewers and consumer_reviews — **my tables**

The file is 242 MB, so this streams it row by row and inserts in chunks instead of loading everything into memory.

`reviewers` is shared by the consumer and expert review tables (schema v3.3). The same name can exist twice, once per `reviewer_type`, because a username and a critic are not the same person. This notebook only creates the `'consumer'` ones. A review with no name gets `reviewer_id = NULL`. Reviewers must exist **before** the reviews that point at them, so each chunk inserts its new names first.

`review_date` comes from the `dateP` column and is filled only if `schema.sql` has that column. `positive_emotion_pct` / `negative_emotion_pct` come from LIWC's `posemo` / `negemo`, and `personal_pronoun_pct` from `ppron`.

`LIMIT` keeps a trial run short. Set it to `None` for everything.

In [15]:
# Author: Paulius Sivickas
LIMIT = None        # e.g. 5000 for a quick trial; None = all rows

# spreadsheet column -> how to turn it into the schema's type
LIWC = [("WC", to_int), ("WPS", num), ("pronoun", num), ("ppron", num),
        ("posemo", num), ("negemo", num), ("Analytic", num), ("Clout", num),
        ("Authentic", num), ("Tone", num), ("certain", num), ("tentat", num)]

INSERT_REVIEWS = """INSERT INTO consumer_reviews
    (movie_id, reviewer_id, review_text, review_score, thumbs_up, total_thumbs,
     word_count, words_per_sentence, pronoun_count, personal_pronoun_pct,
     positive_emotion_pct, negative_emotion_pct, analytic, clout, authentic,
     tone, certain_pct, tentative_pct{extra}) VALUES %s""".format(
        extra=", review_date" if HAVE_REVIEW_DATE else "")

if not HAVE_USER_REVIEWS:
    print("UserReviews file missing - table created but left empty.")
else:
    import openpyxl
    wb = openpyxl.load_workbook(files["user"], read_only=True)
    ws = wb[wb.sheetnames[0]]
    rows = ws.iter_rows(values_only=True)
    hdr  = {name: i for i, name in enumerate(next(rows))}
    g = lambda row, c: row[hdr[c]] if c in hdr and hdr[c] < len(row) else None

    reviewer_ids = {}                         # reviewer name -> reviewer_id
    batch, kept, skipped = [], 0, 0

    def flush(batch):
        """Insert the chunk: new reviewer names first, then the reviews."""
        new_names = sorted({b[1] for b in batch if b[1] and b[1] not in reviewer_ids})
        if new_names:
            got = execute_values(cur,
                "INSERT INTO reviewers (reviewer_name, reviewer_type) VALUES %s "
                "RETURNING reviewer_name, reviewer_id",
                [(n, "consumer") for n in new_names], page_size=5000, fetch=True)
            reviewer_ids.update(dict(got))
        # swap the name (position 1) for its id
        rows_out = [(b[0], reviewer_ids.get(b[1])) + (b[2:] if HAVE_REVIEW_DATE else b[2:-1])
                    for b in batch]
        execute_values(cur, INSERT_REVIEWS, rows_out, page_size=2000)

    for row in rows:
        if LIMIT is not None and kept >= LIMIT:
            break
        mid = url_to_id.get(clean_text(g(row, "url")))
        if mid is None:
            skipped += 1                      # review for a film not in metaClean
            continue
        kept += 1
        batch.append((mid, no_nul(clean_text(g(row, "reviewer"))),
                      no_nul(clean_text(g(row, "Rev"))), num(g(row, "idvscore")),
                      to_int(g(row, "thumbsUp")), to_int(g(row, "thumbsTot")))
                     + tuple(conv(g(row, c)) for c, conv in LIWC)
                     + (parse_date(g(row, "dateP")),))          # last item: review_date
        if len(batch) >= 10000:               # write to the database in chunks
            flush(batch)
            batch = []
    if batch:
        flush(batch)
    wb.close()
    con.commit()

    print(f"reviewers       : {len(reviewer_ids):,}")
    print(f"consumer_reviews: {kept:,} loaded, {skipped:,} skipped (film not in metaClean)")

reviewers       : 106,792
consumer_reviews: 319,662 loaded, 0 skipped (film not in metaClean)


## 10. Check it worked and SQL queries 1-2

Row counts first. `genres`, `movie_genres` and `expert_reviews` are other group members' tables, so they stay empty here (the `expert` reviewers are theirs too).

In [16]:
# Author: Paulius Sivickas
print("ROW COUNTS")
for t in ["movies", "studios", "movie_studios", "people", "movie_credits",
          "reviewers", "consumer_reviews", "genres", "movie_genres", "expert_reviews"]:
    cur.execute(f"SELECT COUNT(*) FROM {t}")
    print(f"  {t:<17} {cur.fetchone()[0]:>10,}")

if HAVE_BILLING:
    cur.execute("SELECT COUNT(*) FROM movie_credits WHERE billing_order IS NOT NULL")
    print(f"\ncast credits with billing_order: {cur.fetchone()[0]:,}")
if HAVE_REVIEW_DATE:
    cur.execute("SELECT COUNT(*), MIN(review_date), MAX(review_date) FROM consumer_reviews WHERE review_date IS NOT NULL")
    print("consumer reviews with review_date (count, first, last):", cur.fetchone())

cur.execute("SELECT COUNT(*) FROM consumer_reviews WHERE reviewer_id IS NULL")
print(f"\nconsumer reviews with no reviewer name (reviewer_id NULL): {cur.fetchone()[0]:,}")

# leftover mojibake (the Ã / Â marker) in names? should be 0 or nearly 0
cur.execute("SELECT COUNT(*) FROM people  WHERE person_name LIKE '%Ã%' OR person_name LIKE '%Â%'")
bad_people = cur.fetchone()[0]
cur.execute("SELECT COUNT(*) FROM studios WHERE studio_name LIKE '%Ã%' OR studio_name LIKE '%Â%'")
bad_studios = cur.fetchone()[0]
print(f"\nnames still garbled: people {bad_people}, studios {bad_studios}")

# QUERY 1: top studios by number of films
q1 = """
SELECT s.studio_name, COUNT(*) AS films,
       ROUND(AVG(m.box_office)) AS avg_box_office
FROM movie_studios ms
JOIN studios s ON s.studio_id = ms.studio_id
JOIN movies  m ON m.movie_id  = ms.movie_id
GROUP BY s.studio_name
ORDER BY films DESC
LIMIT 10
"""
print("\nQUERY 1 - TOP 10 STUDIOS BY NUMBER OF FILMS")
print(pd.read_sql_query(q1, con).to_string(index=False))

# QUERY 2: most-credited actors
q2 = """
SELECT p.person_name, COUNT(*) AS films
FROM movie_credits mc
JOIN people p ON p.person_id = mc.person_id
WHERE mc.credit_type = 'cast'
GROUP BY p.person_name
ORDER BY films DESC
LIMIT 10
"""
print("\nQUERY 2 - TOP 10 MOST-CAST ACTORS")
print(pd.read_sql_query(q2, con).to_string(index=False))

ROW COUNTS
  movies                11,364
  studios                1,118
  movie_studios         11,206
  people                30,314
  movie_credits         59,633
  reviewers            106,792
  consumer_reviews     319,662
  genres                     0
  movie_genres               0
  expert_reviews             0

consumer reviews with no reviewer name (reviewer_id NULL): 3,407

names still garbled: people 1, studios 0

QUERY 1 - TOP 10 STUDIOS BY NUMBER OF FILMS
                           studio_name  films  avg_box_office
                             IFC Films    408       6137236.0
                    Universal Pictures    345     177307667.0
                Sony Pictures Classics    344      12666143.0
                 Warner Bros. Pictures    302     171217280.0
                               Netflix    301       6034728.0
                    Paramount Pictures    280     175212147.0
                     Magnolia Pictures    268       4007551.0
Twentieth Century Fox Film Cor

/var/folders/0y/1vkvn6y96hd9_k1830qdmbvh0000gn/T/ipykernel_27351/1655189562.py:37: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  print(pd.read_sql_query(q1, con).to_string(index=False))
/var/folders/0y/1vkvn6y96hd9_k1830qdmbvh0000gn/T/ipykernel_27351/1655189562.py:50: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  print(pd.read_sql_query(q2, con).to_string(index=False))


## 11. Three more SQL queries (queries 3-5 of 5)

Queries 1-2 are in section 10 (top studios, top cast). These three tie directly to the group's main research question: review characteristics, reviewer agreement, and studio effects. All three use my tables only (`consumer_reviews`, `movie_studios`) plus `movies`, so they do not depend on `expert_reviews`.

In [17]:
# Author: Paulius Sivickas
# Query 3: which films do consumer reviewers agree/disagree on the most?
# Operationalised as the standard deviation of review_score per film (films with >=10 reviews, so two-review ties do not decide the ranking).
q3 = """
SELECT m.movie_id, m.title,
       COUNT(*)                          AS n_reviews,
       ROUND(AVG(cr.review_score), 2)    AS avg_score,
       ROUND(STDDEV(cr.review_score), 2) AS score_stddev
FROM consumer_reviews cr
JOIN movies m ON m.movie_id = cr.movie_id
GROUP BY m.movie_id, m.title
HAVING COUNT(*) >= 10
ORDER BY score_stddev DESC NULLS LAST
LIMIT 15
"""
print("MOST-DISPUTED FILMS (highest review-score spread)")
print(pd.read_sql_query(q3, con).to_string(index=False))

/var/folders/0y/1vkvn6y96hd9_k1830qdmbvh0000gn/T/ipykernel_27351/3072783266.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  print(pd.read_sql_query(q3, con).to_string(index=False))


MOST-DISPUTED FILMS (highest review-score spread)
 movie_id                                     title  n_reviews  avg_score  score_stddev
     5831              My Big Fat Independent Movie         15       4.00          4.83
    10041                            The Undefeated         15       4.00          4.83
     4614                   Justin Bieber's Believe         87       5.00          4.77
     4099            I Hope They Serve Beer in Hell         18       4.11          4.75
     2474                                Dirty Love         32       4.97          4.68
    10992    What the Bleep!?: Down the Rabbit Hole         10       6.10          4.68
     4546 Jonas Brothers: The 3D Concert Experience         16       3.38          4.63
     2180                                Crossroads         11       5.45          4.63
     2346                         Death of a Nation        109       6.09          4.62
     3279                      From Justin to Kelly         13       5

In [18]:
# Author: Paulius Sivickas
# Query 4: does review sentiment (LIWC) track box office?
# Buckets reviews into 10 box-office deciles and averages sentiment/score per bucket.
q4 = """
SELECT decile,
       COUNT(*)                            AS n_reviews,
       ROUND(AVG(review_score), 2)         AS avg_score,
       ROUND(AVG(positive_emotion_pct), 2) AS avg_posemo,
       ROUND(AVG(negative_emotion_pct), 2) AS avg_negemo,
       ROUND(AVG(tone), 2)                 AS avg_tone
FROM (
    SELECT cr.*, NTILE(10) OVER (ORDER BY m.box_office) AS decile
    FROM consumer_reviews cr
    JOIN movies m ON m.movie_id = cr.movie_id
    WHERE m.box_office IS NOT NULL
) sub
GROUP BY decile
ORDER BY decile
"""
print("REVIEW SENTIMENT BY BOX-OFFICE DECILE (1 = lowest grossing, 10 = highest)")
print(pd.read_sql_query(q4, con).to_string(index=False))

REVIEW SENTIMENT BY BOX-OFFICE DECILE (1 = lowest grossing, 10 = highest)


/var/folders/0y/1vkvn6y96hd9_k1830qdmbvh0000gn/T/ipykernel_27351/1644089817.py:21: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  print(pd.read_sql_query(q4, con).to_string(index=False))


 decile  n_reviews  avg_score  avg_posemo  avg_negemo  avg_tone
      1      27368       6.54        5.80        2.72     58.14
      2      27368       6.35        6.25        2.92     59.61
      3      27368       6.33        6.35        2.94     60.21
      4      27368       6.35        6.52        3.04     60.29
      5      27368       6.38        6.66        2.84     62.36
      6      27368       6.41        6.88        2.72     64.49
      7      27368       6.58        6.62        2.56     64.32
      8      27368       6.92        6.84        2.44     66.63
      9      27368       7.15        6.66        2.42     66.78
     10      27367       7.35        6.59        2.40     64.79


In [19]:
# Author: Paulius Sivickas
# Query 5: co-productions vs single-studio films (uses the movie_studios bridge table)
q5 = """
SELECT CASE WHEN studio_count = 0 THEN 'no studio listed'
            WHEN studio_count = 1 THEN 'single studio'
            ELSE 'co-production' END      AS category,
       COUNT(*)                           AS n_movies,
       ROUND(AVG(box_office), 0)          AS avg_box_office,
       ROUND(AVG(production_budget), 0)   AS avg_production_budget
FROM (
    SELECT m.movie_id, m.box_office, m.production_budget,
           COUNT(ms.studio_id) AS studio_count
    FROM movies m
    LEFT JOIN movie_studios ms ON ms.movie_id = m.movie_id
    GROUP BY m.movie_id, m.box_office, m.production_budget
) per_movie
GROUP BY category
ORDER BY n_movies DESC
"""
print("CO-PRODUCTIONS VS SINGLE-STUDIO FILMS")
print(pd.read_sql_query(q5, con).to_string(index=False))

CO-PRODUCTIONS VS SINGLE-STUDIO FILMS
        category  n_movies  avg_box_office  avg_production_budget
   single studio     10822      78215794.0             40895758.0
no studio listed       350      22957759.0             22262000.0
   co-production       192      41402311.0             22471519.0


/var/folders/0y/1vkvn6y96hd9_k1830qdmbvh0000gn/T/ipykernel_27351/2310181420.py:21: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  print(pd.read_sql_query(q5, con).to_string(index=False))


## 12. Python encapsulator (5 functions)

Each function hides its SQL behind a plain Python call and returns a ready-to-use DataFrame for the statistics/ML work in the other modules. The first two wrap queries 1-2 (section 10); the last three wrap queries 3-5 (section 11).

In [20]:
# Author: Paulius Sivickas

def get_top_studios(con, n=10):
    """Studios with the most films, and their average box office."""
    q = """
        SELECT s.studio_name, COUNT(*) AS films,
               ROUND(AVG(m.box_office), 0) AS avg_box_office
        FROM movie_studios ms
        JOIN studios s ON s.studio_id = ms.studio_id
        JOIN movies  m ON m.movie_id  = ms.movie_id
        GROUP BY s.studio_name
        ORDER BY films DESC
        LIMIT %s
    """
    return pd.read_sql_query(q, con, params=(n,))

def get_top_cast(con, n=10):
    """The people credited as cast in the most films."""
    q = """
        SELECT p.person_name, COUNT(*) AS films
        FROM movie_credits mc
        JOIN people p ON p.person_id = mc.person_id
        WHERE mc.credit_type = 'cast'
        GROUP BY p.person_name
        ORDER BY films DESC
        LIMIT %s
    """
    return pd.read_sql_query(q, con, params=(n,))

In [21]:
# Author: Paulius Sivickas

def get_review_score_spread(con, min_reviews=10, n=15):
    """Films where consumer reviewers disagree with each other the most
    (highest standard deviation in review_score). This is the "agreement"
    measure the MRQ asks about."""
    q = """
        SELECT m.movie_id, m.title,
               COUNT(*) AS n_reviews,
               ROUND(AVG(cr.review_score), 2) AS avg_score,
               ROUND(STDDEV(cr.review_score), 2) AS score_stddev
        FROM consumer_reviews cr
        JOIN movies m ON m.movie_id = cr.movie_id
        GROUP BY m.movie_id, m.title
        HAVING COUNT(*) >= %s
        ORDER BY score_stddev DESC NULLS LAST
        LIMIT %s
    """
    return pd.read_sql_query(q, con, params=(min_reviews, n))

def get_sentiment_by_box_office_decile(con, n_deciles=10):
    """Average review sentiment (LIWC) and score, grouped by box-office decile.
    Tests whether review characteristics track box office."""
    q = """
        SELECT decile,
               COUNT(*) AS n_reviews,
               ROUND(AVG(review_score), 2) AS avg_score,
               ROUND(AVG(positive_emotion_pct), 2) AS avg_posemo,
               ROUND(AVG(negative_emotion_pct), 2) AS avg_negemo,
               ROUND(AVG(tone), 2) AS avg_tone
        FROM (
            SELECT cr.*, NTILE(%s) OVER (ORDER BY m.box_office) AS decile
            FROM consumer_reviews cr
            JOIN movies m ON m.movie_id = cr.movie_id
            WHERE m.box_office IS NOT NULL
        ) sub
        GROUP BY decile
        ORDER BY decile
    """
    return pd.read_sql_query(q, con, params=(n_deciles,))

def get_coproduction_vs_single_studio(con):
    """Compares films with no studio listed, one studio, and co-productions
    (movie_studios bridge table) on average box office and budget."""
    q = """
        SELECT CASE WHEN studio_count = 0 THEN 'no studio listed'
                    WHEN studio_count = 1 THEN 'single studio'
                    ELSE 'co-production' END AS category,
               COUNT(*) AS n_movies,
               ROUND(AVG(box_office), 0) AS avg_box_office,
               ROUND(AVG(production_budget), 0) AS avg_production_budget
        FROM (
            SELECT m.movie_id, m.box_office, m.production_budget,
                   COUNT(ms.studio_id) AS studio_count
            FROM movies m
            LEFT JOIN movie_studios ms ON ms.movie_id = m.movie_id
            GROUP BY m.movie_id, m.box_office, m.production_budget
        ) per_movie
        GROUP BY category
        ORDER BY n_movies DESC
    """
    return pd.read_sql_query(q, con)

In [22]:
# Author: Paulius Sivickas
# Quick demo that each encapsulator function returns a usable DataFrame.
print("get_top_studios(con, 5):")
print(get_top_studios(con, 5))
print()
print("get_top_cast(con, 5):")
print(get_top_cast(con, 5))
print()
print("get_review_score_spread(con, min_reviews=10, n=5):")
print(get_review_score_spread(con, min_reviews=10, n=5))
print()
print("get_sentiment_by_box_office_decile(con):")
print(get_sentiment_by_box_office_decile(con))
print()
print("get_coproduction_vs_single_studio(con):")
print(get_coproduction_vs_single_studio(con))

get_top_studios(con, 5):
              studio_name  films  avg_box_office
0               IFC Films    408       6137236.0
1      Universal Pictures    345     177307667.0
2  Sony Pictures Classics    344      12666143.0
3   Warner Bros. Pictures    302     171217280.0
4                 Netflix    301       6034728.0

get_top_cast(con, 5):
         person_name  films
0  Samuel L. Jackson     47
1     Morgan Freeman     45
2       Nicolas Cage     43
3        Liam Neeson     42
4     Julianne Moore     42

get_review_score_spread(con, min_reviews=10, n=5):
   movie_id                           title  n_reviews  avg_score  \
0      5831    My Big Fat Independent Movie         15       4.00   
1     10041                  The Undefeated         15       4.00   
2      4614         Justin Bieber's Believe         87       5.00   
3      4099  I Hope They Serve Beer in Hell         18       4.11   
4      2474                      Dirty Love         32       4.97   

   score_stddev  
0    

/var/folders/0y/1vkvn6y96hd9_k1830qdmbvh0000gn/T/ipykernel_27351/1346087928.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(q, con, params=(n,))
/var/folders/0y/1vkvn6y96hd9_k1830qdmbvh0000gn/T/ipykernel_27351/1346087928.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(q, con, params=(n,))
/var/folders/0y/1vkvn6y96hd9_k1830qdmbvh0000gn/T/ipykernel_27351/995401468.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(q, con, params=(min_review

   decile  n_reviews  avg_score  avg_posemo  avg_negemo  avg_tone
0       1      27368       6.54        5.80        2.72     58.14
1       2      27368       6.35        6.25        2.92     59.61
2       3      27368       6.33        6.35        2.94     60.21
3       4      27368       6.35        6.52        3.04     60.29
4       5      27368       6.38        6.66        2.84     62.36
5       6      27368       6.41        6.88        2.72     64.49
6       7      27368       6.58        6.62        2.56     64.32
7       8      27368       6.92        6.84        2.44     66.63
8       9      27368       7.15        6.66        2.42     66.78
9      10      27367       7.35        6.59        2.40     64.79

get_coproduction_vs_single_studio(con):
           category  n_movies  avg_box_office  avg_production_budget
0     single studio     10822      78215794.0             40895758.0
1  no studio listed       350      22957759.0             22262000.0
2     co-production       

/var/folders/0y/1vkvn6y96hd9_k1830qdmbvh0000gn/T/ipykernel_27351/995401468.py:62: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(q, con)


In [23]:
# Author: Paulius Sivickas
# Export the cleaned tables to CSV, so the group can use them without the huge spreadsheets.
# Run this AFTER the load cells above. Files land in the folder "cleaned_data" next to this notebook.
# Big tables are cut into parts of 50,000 rows, so every file stays far below GitHub's 100 MB limit.
import os, math

EXPORT_DIR = "cleaned_data"
PART_ROWS = 50_000
os.makedirs(EXPORT_DIR, exist_ok=True)

# table -> columns that give a fixed order, so the parts never overlap
EXPORT = {
    "movies":           "movie_id",
    "studios":          "studio_id",
    "movie_studios":    "movie_id, studio_id",
    "people":           "person_id",
    "movie_credits":    "movie_id, person_id, credit_type",
    "reviewers":        "reviewer_id",
    "consumer_reviews": "review_id",
}

for table, order_by in EXPORT.items():
    cur.execute(f"SELECT COUNT(*) FROM {table}")
    n_rows = cur.fetchone()[0]
    n_parts = max(1, math.ceil(n_rows / PART_ROWS))
    for p in range(n_parts):
        name = f"{table}.csv" if n_parts == 1 else f"{table}_part{p + 1}.csv"
        path = os.path.join(EXPORT_DIR, name)
        sql = (f"COPY (SELECT * FROM {table} ORDER BY {order_by} "
               f"LIMIT {PART_ROWS} OFFSET {p * PART_ROWS}) "
               f"TO STDOUT WITH (FORMAT CSV, HEADER)")
        with open(path, "w", encoding="utf-8", newline="") as f:
            cur.copy_expert(sql, f)
        print(f"{name:<32} {os.path.getsize(path) / 1e6:7.1f} MB")
    print(f"  -> {table}: {n_rows:,} rows in {n_parts} file(s)")


movies.csv                           1.0 MB
  -> movies: 11,364 rows in 1 file(s)
studios.csv                          0.0 MB
  -> studios: 1,118 rows in 1 file(s)
movie_studios.csv                    0.1 MB
  -> movie_studios: 11,206 rows in 1 file(s)
people.csv                           0.6 MB
  -> people: 30,314 rows in 1 file(s)
movie_credits_part1.csv              0.8 MB
movie_credits_part2.csv              0.2 MB
  -> movie_credits: 59,633 rows in 2 file(s)
reviewers_part1.csv                  1.2 MB
reviewers_part2.csv                  1.2 MB
reviewers_part3.csv                  0.2 MB
  -> reviewers: 106,792 rows in 3 file(s)
consumer_reviews_part1.csv          39.2 MB
consumer_reviews_part2.csv          36.9 MB
consumer_reviews_part3.csv          40.9 MB
consumer_reviews_part4.csv          46.7 MB
consumer_reviews_part5.csv          39.9 MB
consumer_reviews_part6.csv          32.5 MB
consumer_reviews_part7.csv          12.3 MB
  -> consumer_reviews: 319,662 rows in 7 file(s)


In [24]:
# Author: Paulius Sivickas
con.commit()
con.close()
print("finished")

finished
